# POC 03 — Build Customer 360
Attach `lh_customer_360` as the default Lakehouse and upload the three CSV files to `Files/raw`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
    DecimalType, DateType, TimestampType)

crm_schema = StructType([
    StructField('crm_customer_id', StringType(), False),
    StructField('full_name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('city', StringType(), True),
    StructField('segment', StringType(), True),
    StructField('updated_at', TimestampType(), True),
])
order_schema = StructType([
    StructField('order_id', StringType(), False),
    StructField('customer_email', StringType(), True),
    StructField('order_date', DateType(), True),
    StructField('amount', DecimalType(12, 2), True),
    StructField('status', StringType(), True),
])
ticket_schema = StructType([
    StructField('ticket_id', StringType(), False),
    StructField('contact_email', StringType(), True),
    StructField('created_at', DateType(), True),
    StructField('priority', StringType(), True),
    StructField('status', StringType(), True),
])

## Bronze — preserve each source contract

In [ ]:
def read_csv(name, schema):
    return (spark.read.option('header', True).schema(schema)
        .csv(f'Files/raw/{name}.csv')
        .withColumn('_source_file', F.input_file_name())
        .withColumn('_ingested_at', F.current_timestamp()))

bronze_crm = read_csv('crm_customers', crm_schema)
bronze_orders = read_csv('commerce_orders', order_schema)
bronze_tickets = read_csv('support_tickets', ticket_schema)

for name, frame in {
    'bronze_crm_customers': bronze_crm,
    'bronze_commerce_orders': bronze_orders,
    'bronze_support_tickets': bronze_tickets,
}.items():
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(name)

## Silver — normalize identity and validate records
Email is the deterministic matching key for this POC. Unmatched records remain visible for stewardship.

In [ ]:
def normalize_email(column):
    return F.lower(F.trim(column))

latest_crm = Window.partitionBy('normalized_email').orderBy(F.col('updated_at').desc())
silver_crm = (bronze_crm
    .withColumn('normalized_email', normalize_email(F.col('email')))
    .filter(F.col('crm_customer_id').isNotNull() & F.col('normalized_email').contains('@'))
    .withColumn('_rank', F.row_number().over(latest_crm))
    .filter(F.col('_rank') == 1).drop('_rank'))

silver_orders = (bronze_orders
    .withColumn('normalized_email', normalize_email(F.col('customer_email')))
    .filter(F.col('order_id').isNotNull() & F.col('order_date').isNotNull()
        & (F.col('amount') >= 0) & F.col('status').isin('Completed', 'Pending', 'Cancelled'))
    .dropDuplicates(['order_id']))

silver_tickets = (bronze_tickets
    .withColumn('normalized_email', normalize_email(F.col('contact_email')))
    .filter(F.col('ticket_id').isNotNull() & F.col('created_at').isNotNull()
        & F.col('status').isin('Open', 'Closed'))
    .dropDuplicates(['ticket_id']))

for name, frame in {
    'silver_crm_customers': silver_crm,
    'silver_commerce_orders': silver_orders,
    'silver_support_tickets': silver_tickets,
}.items():
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(name)

## Gold — governed dimension, facts, and unmatched queues

In [ ]:
dim_customer = (silver_crm
    .withColumn('customer_key', F.sha2('normalized_email', 256))
    .select('customer_key', 'crm_customer_id', 'full_name', 'normalized_email',
        'city', 'segment', 'updated_at'))

unmatched_orders = silver_orders.join(
    dim_customer.select('normalized_email'), 'normalized_email', 'left_anti')
unmatched_tickets = silver_tickets.join(
    dim_customer.select('normalized_email'), 'normalized_email', 'left_anti')

fact_sales = (silver_orders.filter(F.col('status') == 'Completed')
    .join(dim_customer.select('customer_key', 'normalized_email'), 'normalized_email', 'inner')
    .select('order_id', 'customer_key', 'order_date', 'amount', 'status'))
fact_support = (silver_tickets
    .join(dim_customer.select('customer_key', 'normalized_email'), 'normalized_email', 'inner')
    .select('ticket_id', 'customer_key', 'created_at', 'priority', 'status'))

for name, frame in {
    'dim_customer': dim_customer, 'fact_sales': fact_sales,
    'fact_support': fact_support, 'unmatched_orders': unmatched_orders,
    'unmatched_tickets': unmatched_tickets,
}.items():
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(name)

## Customer 360 — one governed row per customer

In [ ]:
sales_by_customer = fact_sales.groupBy('customer_key').agg(
    F.countDistinct('order_id').alias('total_orders'),
    F.sum('amount').alias('lifetime_value'),
    F.max('order_date').alias('last_order_date'))
support_by_customer = (fact_support.filter(F.col('status') == 'Open')
    .groupBy('customer_key').agg(F.countDistinct('ticket_id').alias('open_tickets')))

gold_customer_360 = (dim_customer
    .join(sales_by_customer, 'customer_key', 'left')
    .join(support_by_customer, 'customer_key', 'left')
    .withColumn('total_orders', F.coalesce('total_orders', F.lit(0)))
    .withColumn('lifetime_value', F.coalesce('lifetime_value', F.lit(0).cast('decimal(12,2)')))
    .withColumn('open_tickets', F.coalesce('open_tickets', F.lit(0)))
    .withColumn('value_band',
        F.when(F.col('lifetime_value') >= 1000, 'High value')
         .when(F.col('lifetime_value') >= 500, 'Established')
         .otherwise('Developing'))
    .select('customer_key', 'crm_customer_id', 'full_name', 'segment', 'city',
        'total_orders', 'lifetime_value', 'last_order_date', 'open_tickets', 'value_band'))

gold_customer_360.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold_customer_360')
display(gold_customer_360.orderBy('full_name'))
print(f'Matched revenue: {fact_sales.agg(F.sum("amount")).first()[0]}')
print(f'Unmatched orders: {unmatched_orders.count()}; unmatched tickets: {unmatched_tickets.count()}')